# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/furkankumrudev/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
%pip -q install duckdb

import os
import duckdb
from google.colab import userdata

# Hugging Face token'ı Colab Secrets'tan alınıyor
HF_TOKEN = userdata.get("HF_TOKEN")

# DuckDB bağlantısı başlatılıyor ('con' değişkeni burada tanımlanır)
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connection ready.")

DuckDB connection ready.


## 1. My rule and its reason codes

**The Rule Idea (Quick Win / Striking Distance):**
A content piece is worth optimizing immediately if it generates a high volume of impressions (visibility) but sits in "striking distance" (average position between 11 and 20, roughly Page 2) with a very low Click-Through Rate (CTR). By tweaking the title or meta description, a small rank increase can yield massive traffic.

- **Action Label:** `CTR_Fix_or_Refresh`
- **Reason Code:** `visible_page2_underperforming`

**Signal Verdicts:**
1. **CTR-vs-Position:** `CONFIRMED`. As average position drops from Page 1 to Page 2, CTR decreases significantly. This is a real, measurable signal.
2. **Volume / Impressions:** `CONFIRMED`. A very small subset of high-volume pages drives the majority of the data. High-impression filtering is essential to prioritize effort.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. THE SIGNAL CHECKS
REL = "hf://datasets/FlyRank/internship-warehouse"
fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Signal 1: CTR vs Position Bucket
signal_ctr = con.sql(f"""
SELECT
    CASE
        WHEN gsc_avg_position <= 3 THEN '1. Top 3'
        WHEN gsc_avg_position <= 10 THEN '2. Page 1 (4-10)'
        WHEN gsc_avg_position <= 20 THEN '3. Page 2 (11-20)'
        ELSE '4. Page 3+'
    END AS position_bucket,
    COUNT(*) AS n,
    ROUND(AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)), 4) AS avg_ctr
FROM {fact_daily}
WHERE gsc_avg_position > 0 AND gsc_impressions > 0
GROUP BY 1
ORDER BY 1
""").df()

print("Signal 1: CTR vs Position Profile (Verdict: CONFIRMED)")
display(signal_ctr)

# Signal 2: Volume / Impressions Bucket
signal_vol = con.sql(f"""
SELECT
    CASE
        WHEN gsc_impressions < 100 THEN '1. Low (<100)'
        WHEN gsc_impressions < 1000 THEN '2. Med (100-1k)'
        ELSE '3. High (>1k)'
    END AS volume_bucket,
    COUNT(*) AS n,
    SUM(gsc_clicks) AS total_clicks
FROM {fact_daily}
WHERE gsc_data_available IS TRUE
GROUP BY 1
ORDER BY 1
""").df()

print("\nSignal 2: Volume Distribution (Verdict: CONFIRMED)")
display(signal_vol)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1: CTR vs Position Profile (Verdict: CONFIRMED)


,position_bucket,n,avg_ctr
0,1. Top 3,564173,0.0049
1,2. Page 1 (4-10),1456122,0.0035
2,3. Page 2 (11-20),519223,0.0028
3,4. Page 3+,908354,0.0013


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Signal 2: Volume Distribution (Verdict: CONFIRMED)


,volume_bucket,n,total_clicks
0,1. Low (<100),2972453,172298.0
1,2. Med (100-1k),606189,485209.0
2,3. High (>1k),32419,164325.0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import numpy as np

# Aggregate March 2026 data per content item
df_content = con.sql(f"""
SELECT
    content_hash_id,
    MAX(client_hash_id) AS client_hash_id,
    SUM(gsc_impressions) AS total_impressions,
    SUM(gsc_clicks) AS total_clicks,
    AVG(gsc_avg_position) AS avg_pos
FROM {fact_daily}
WHERE gsc_data_available IS TRUE
GROUP BY 1
HAVING SUM(gsc_impressions) > 0
""").df()

# 1. Base conditions (Transparent 0 or 1 flags)
visible = (df_content["total_impressions"] >= 1000).astype(int)
striking_dist = ((df_content["avg_pos"] > 10) & (df_content["avg_pos"] <= 20)).astype(int)
low_ctr = ((df_content["total_clicks"] / df_content["total_impressions"]) < 0.015).astype(int)

# 2. Transparent Score Calculation (No fitted weights)
df_content["score"] = visible * striking_dist * low_ctr * df_content["total_impressions"]

# 3. Action and Reason Codes
df_content["action_label"] = np.where(df_content["score"] > 0, "CTR_Fix_or_Refresh", "None")
df_content["reason_code"] = np.where(df_content["score"] > 0, "visible_page2_underperforming", "none")

# 4. Filter and Rank the queue
df_ranked = df_content[df_content["score"] > 0].sort_values(by="score", ascending=False).copy()

# 5. Write to CSV
os.makedirs("work/outputs", exist_ok=True)
csv_path = "work/outputs/baseline_action_score.csv"
df_ranked.to_csv(csv_path, index=False)

print(f"Ranked queue written to {csv_path}. Total actionable items: {len(df_ranked)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked queue written to work/outputs/baseline_action_score.csv. Total actionable items: 7467


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Display Top 10 for manual review
top_10 = df_ranked.head(10).reset_index(drop=True)
display(top_10)

,content_hash_id,client_hash_id,total_impressions,total_clicks,avg_pos,score,action_label,reason_code
0,content_e8a52cf3d5988c07,client_23a62021009f63c4,244931.0,669.0,15.008339,244931.0,CTR_Fix_or_Refresh,visible_page2_underperforming
1,content_66288edeb93b7c4f,client_23a62021009f63c4,137878.0,782.0,18.615742,137878.0,CTR_Fix_or_Refresh,visible_page2_underperforming
2,content_5e1c049f62e33b11,client_23a62021009f63c4,120175.0,168.0,18.077081,120175.0,CTR_Fix_or_Refresh,visible_page2_underperforming
3,content_9c057b66c30a3abb,client_73cda7b4e4f265ea,83834.0,1.0,11.195379,83834.0,CTR_Fix_or_Refresh,visible_page2_underperforming
4,content_f6723f0229e1bfdc,client_23a62021009f63c4,69822.0,13.0,15.766691,69822.0,CTR_Fix_or_Refresh,visible_page2_underperforming
5,content_65c75874a23fca87,client_23a62021009f63c4,64935.0,18.0,13.571837,64935.0,CTR_Fix_or_Refresh,visible_page2_underperforming
6,content_f0703fc6ae385591,client_a80fca3f171ed1de,63201.0,2.0,10.060535,63201.0,CTR_Fix_or_Refresh,visible_page2_underperforming
7,content_2690f62f39fb14fe,client_23a62021009f63c4,61074.0,101.0,17.739450,61074.0,CTR_Fix_or_Refresh,visible_page2_underperforming
8,content_fd74df47e4e3ffc6,client_20259bd6705d81d4,55091.0,164.0,10.842341,55091.0,CTR_Fix_or_Refresh,visible_page2_underperforming
9,content_67b87ba1ac3d0798,client_20259bd6705d81d4,54505.0,56.0,10.468723,54505.0,CTR_Fix_or_Refresh,visible_page2_underperforming


### Top-10 Manual Review

1. **Rank 1**: Action: `CTR_Fix_or_Refresh` | Reason: `visible_page2_underperforming` | *Wrong if:* The high volume comes from generic, irrelevant search terms where ranking higher won't drive qualified traffic anyway.
2. **Rank 2**: Action: `CTR_Fix_or_Refresh` | Reason: `visible_page2_underperforming` | *Wrong if:* The page is an image gallery or an informational snippet where users get their answer directly on the Google SERP without clicking.
3. **Rank 3**: Action: `CTR_Fix_or_Refresh` | Reason: `visible_page2_underperforming` | *Wrong if:* The client recently unpublished or heavily redirected the page late in the month.
4. **Rank 4**: Action: `CTR_Fix_or_Refresh` | Reason: `visible_page2_underperforming` | *Wrong if:* The content is tied to a specific past event (e.g., "2025 conference") that naturally lost relevance.
5. **Rank 5**: Action: `CTR_Fix_or_Refresh` | Reason: `visible_page2_underperforming` | *Wrong if:* The low CTR is strictly due to intense branded search dominance by a competitor above them.
6. **Rank 6**: Action: `CTR_Fix_or_Refresh` | Reason: `visible_page2_underperforming` | *Wrong if:* Our "average" position of 15 is actually heavily skewed by ranking #50 for most of the month and jumping to #1 in the final two days.
7. **Rank 7**: Action: `CTR_Fix_or_Refresh` | Reason: `visible_page2_underperforming` | *Wrong if:* The page is intentionally a low-click "contact info" directory.
8. **Rank 8**: Action: `CTR_Fix_or_Refresh` | Reason: `visible_page2_underperforming` | *Wrong if:* The impressions spiked due to a brief viral trend that has already completely died down.
9. **Rank 9**: Action: `CTR_Fix_or_Refresh` | Reason: `visible_page2_underperforming` | *Wrong if:* The page has a technical SEO error (like a canonical tag issue) that makes a simple text refresh useless.
10. **Rank 10**: Action: `CTR_Fix_or_Refresh`| Reason: `visible_page2_underperforming` | *Wrong if:* The page ranks on Page 2 for a completely unrelated query due to Google misinterpreting a single keyword.

## 4. Weak picks + leakage check

**Weak Picks Identified:**
As noted in the manual review, some top picks might be "false positives". Because our rule relies on an aggregate `AVG(gsc_avg_position)` over the entire month, a page could have ranked #50 for three weeks, then successfully jumped to #1 for one week. Its `avg_pos` might look like 15 (Page 2), tricking our rule into thinking it's stuck on Page 2, when in reality the SEO strategy is already working.

**Leakage Check:**
- I strictly used historical metrics (`gsc_impressions`, `gsc_clicks`, `gsc_avg_position`) from the isolated evaluation month (March 2026).
- I did not include any target-derived labels (`is_declining_label`, `trend_pct`, `trend_direction`).
- There are no future windows or hidden flags leaking into the baseline logic. The score is 100% transparent and decision-time safe.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.